# Full Bronze tripdata Processing and submition to silver layer

In [0]:
import re
import time
import os
from collections import defaultdict
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

In [0]:
storage_name = os.environ["STORAGE_ACCOUNT_NAME"]
storage_key = os.environ["STORAGE_ACCOUNT_KEY"]

bronze_base = f"abfss://bronze@{storage_name}.dfs.core.windows.net/historical_tripdata"
silver_base = f"abfss://silver@{storage_name}.dfs.core.windows.net/historical_tripdata"

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_name}.dfs.core.windows.net", storage_key
)

# 🛡️ Silver Layer Transformation: 5-Point EDA Hygiene Protocol

This processing block enforces our Medallion Lakehouse architecture by transforming raw, string-based Bronze CSV records into a fully typed, ACID-compliant Silver schema. Every transformation rule below is directly derived from our local exploratory data forensics:

### Core Engineering Decisions:

1. **Flexible ISO-8601 Timestamp Parsing:** * *Forensic Discovery:* Historical CSVs mix standard second precision (`:ss`) with Next-Gen e-bike millisecond telemetry (`.SSS`). Hardcoded regex format strings cause fatal JVM parsing crashes (`SQLSTATE: 22007`).
   * *Decision:* Stripped explicit format strings to allow PySpark's native ISO engine to dynamically absorb mixed precisions.
2. **Spatial Standardization (6-Decimal Precision):** * *Decision:* Explicitly cast raw GPS coordinates to `DOUBLE` and rounded to standard shared-mobility precision (`0.000001`), enabling highly accurate physical dock mapping and downstream spatial clustering (K-Means).
3. **Safe Imputation ("Valet & Dockless Preservation"):** * *Forensic Discovery:* Missing *End* stations occur 5x more frequently than Start stations due to dockless e-bike street locking. Furthermore, major events utilize handheld "Valet Corrals" that generate Station Names without Station IDs.
   * *Decision:* Wrapped station attributes in `coalesce()` to assign explicit `"UNASSIGNED"` / `"UNKNOWN"` flags. This prevents downstream Gold-layer inner joins from silently wiping out legitimate revenue-generating rides.
4. **Pre-Extracted Temporal Partitions:** * *Decision:* Materialized `start_year`, `start_month`, `start_date`, and `start_hour` at the Silver staging level to eliminate expensive runtime timestamp casting during heavy BI logistics aggregations.
5. **The Iron Guardrails (Time & Geo-Fencing):**
   * *The Time Fence:* Restricted valid trips strictly between **60 seconds** (purging false-start accidental clicks) and **86,400 seconds / 24 hours** (purging 25-hour open-dock system auto-timeouts).
   * *The Geo-Fence:* Clamped coordinates inside the NYC Metro box `[Lat: 40.4 to 41.0, Lng: -74.3 to -73.7]`, successfully assassinating default `(0.0, 0.0)` "Null Island" hardware dropouts.

In [0]:
def clean_bronze_trips_to_silver(df_raw):
    """Our proven 5-point local hygiene protocol"""
    return (
        df_raw
        .withColumn("started_at_ts", F.to_timestamp("started_at"))
        .withColumn("ended_at_ts", F.to_timestamp("ended_at"))
        .withColumn("trip_duration_seconds", F.unix_timestamp("ended_at_ts") - F.unix_timestamp("started_at_ts"))
        .withColumn("start_lat", F.round(F.col("start_lat").cast(DoubleType()), 6))
        .withColumn("start_lng", F.round(F.col("start_lng").cast(DoubleType()), 6))
        .withColumn("end_lat", F.round(F.col("end_lat").cast(DoubleType()), 6))
        .withColumn("end_lng", F.round(F.col("end_lng").cast(DoubleType()), 6))
        .withColumn("start_station_id", F.coalesce(F.col("start_station_id"), F.lit("UNASSIGNED")))
        .withColumn("start_station_name", F.coalesce(F.col("start_station_name"), F.lit("UNKNOWN")))
        .withColumn("end_station_id", F.coalesce(F.col("end_station_id"), F.lit("UNASSIGNED")))
        .withColumn("end_station_name", F.coalesce(F.col("end_station_name"), F.lit("UNKNOWN")))
        .withColumn("start_year", F.year("started_at_ts"))
        .withColumn("start_month", F.month("started_at_ts"))
        .withColumn("start_date", F.to_date("started_at_ts"))
        .withColumn("start_hour", F.hour("started_at_ts"))
        .filter(
            (F.col("trip_duration_seconds") >= 60) & (F.col("trip_duration_seconds") <= 86400) & 
            (F.col("start_lat").between(40.4, 41.0)) & (F.col("start_lng").between(-74.3, -73.7)) & 
            (F.col("end_lat").between(40.4, 41.0)) & (F.col("end_lng").between(-74.3, -73.7))
        )
        .drop("started_at", "ended_at")
    )

In [0]:
# ==========================================
# STEP 1: FORENSIC DATA LAKE SCANNER
# ==========================================
print("Scanning ADLS Gen2 Bronze Container for trip segments...")
raw_file_metadata = dbutils.fs.ls(bronze_base)

# Map every file path to its YYYYMM prefix
monthly_bundles = defaultdict(list)

for file_obj in raw_file_metadata:
    if file_obj.name.endswith(".csv"):
        # Extracts '202407' out of 'abfss://.../202407-citibike-tripdata_2.csv'
        match = re.search(r"/(\d{6})-citibike", file_obj.path)
        if match:
            ym_key = match.group(1)
            monthly_bundles[ym_key].append(file_obj.path)

sorted_months = sorted(monthly_bundles.keys())
total_files = sum(len(v) for v in monthly_bundles.values())

print(f"--> Discovered {total_files} CSV segments mapped across {len(sorted_months)} distinct monthly batches.\n")

In [0]:
# ==========================================
# STEP 2: THE CONVEYOR BELT EXECUTION
# ==========================================
overall_start_time = time.time()

for i, ym in enumerate(sorted_months, 1):
    batch_start = time.time()
    file_list = monthly_bundles[ym]
    
    print(f"[{i}/{len(sorted_months)}] Ingesting {ym} (Merging {len(file_list)} split CSVs)...", end=" ")
    
    # 1. Read just this specific month's file bundle
    df_month_raw = spark.read.option("header", "true").csv(file_list)
    
    # 2. Transform
    df_month_clean = clean_bronze_trips_to_silver(df_month_raw)
    
    # 3. Append into Delta Lake natively
    df_month_clean.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("start_year", "start_month") \
        .save(silver_base)
        
    # 4. FORCE JVM GARBAGE COLLECTION: Wipe the DAG from cluster RAM
    spark.catalog.clearCache()
    
    batch_elapsed = time.time() - batch_start
    print(f"✓ Committed to Delta in {batch_elapsed:.1f}s")

total_time = (time.time() - overall_start_time) / 60
print(f"\n*** TRIUMPH: All {total_files} files materialized to Silver Delta Lake in {total_time:.2f} minutes! ***")